### 4. Untuk File: `01_audit_dan_visualisasi_awal.py` (atau `.ipynb`)
**Lokasi:** Di dalam folder `notebooks/` (`proyek-A2-icecube/notebooks/01_audit_dan_visualisasi_awal.py`)
**Fungsi:** Kode Python utama Minggu 1 untuk mencatat versi paket, membuat *checksum* SHA-256 data mentah, mengaudit data (Langkah 1), mengonversi koordinat ke galaktik, dan menghasilkan **3 gambar awal wajib Minggu 1** (Langkah 2).

In [6]:
import sys
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import astropy
import matplotlib
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord
import astropy.units as u

# ==========================================
# 0. PENCATATAN VERSI LINGKUNGAN (WAJIB)
# ==========================================
print("Python version:", sys.version)
for mod in (np, scipy, pd, astropy, matplotlib):
    print(f"{mod.__name__}: {mod.__version__}")

# ==========================================
# 1. PEMERIKSAAN CHECKSUM SHA-256 DATA MENTAH
# ==========================================
root_dir = Path("..") if Path.cwd().name.lower() == "notebooks" else Path(".")
raw_dir = root_dir / "dataverse_files"
log_dir = root_dir / "Logs"
fig_dir = root_dir / "Results" / "Figures"
tab_dir = root_dir / "Results" / "Tables"

log_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)
tab_dir.mkdir(parents=True, exist_ok=True)

files = sorted((raw_dir / "events").glob("*.csv"))
if not files:
    print("PERINGATAN: Belum ada berkas data di dataverse_files/events/.")
else:
    with open(log_dir / "checksum_data_mentah.txt", "w") as f_out:
        for f in files:
            sha256 = hashlib.sha256(f.read_bytes()).hexdigest()
            f_out.write(f"{sha256}  {f.name}\n")
    print(f"Checksum {len(files)} berkas berhasil disimpan ke Logs/checksum_data_mentah.txt")

# ==========================================
# 2. PEMUATAN & AUDIT DATA (LANGKAH 1)
# ==========================================
# CATATAN: Sesuaikan daftar KOLOM dengan README dataset PS-IC40-IC86_VII setelah Gerbang 0!
KOLOM = ["mjd", "log10E", "ang_err", "ra", "dec", "azi", "zen"]

frames = []
for f in sorted(raw_dir.glob("events/*.csv")):  # Sesuaikan path subfolder jika berbeda
    df = pd.read_csv(f, sep=r"\s+", comment="#", header=None, names=KOLOM)
    df["musim"] = f.stem
    frames.append(df)

if frames:
    ev = pd.concat(frames, ignore_index=True)
    print("\n--- RINGKASAN AUDIT DATA ---")
    print("Total baris & kolom:", ev.shape)
    print("\nStatistik deskriptif:\n", ev.describe())
    print("\nNilai hilang (NaN):\n", ev.isna().sum())
    print("\nJumlah duplikat (mjd, ra, dec):", ev.duplicated(subset=["mjd", "ra", "dec"]).sum())

    # Uji batas fisis
    assert ev["ra"].between(0.0, 360.0).all(), "Ada RA di luar [0, 360)!"
    assert ev["dec"].between(-90.0, 90.0).all(), "Ada Deklinasi di luar [-90, 90]!"
    assert (ev["ang_err"] > 0).all(), "Ada galat sudut <= 0!"

    ringkasan_musim = ev.groupby("musim").agg(
        jumlah_event=("ra", "size"),
        mjd_min=("mjd", "min"),
        mjd_max=("mjd", "max"),
        median_log10E=("log10E", "median"),
        median_ang_err=("ang_err", "median"),
    )
    print("\nRingkasan per musim:\n", ringkasan_musim)
    ringkasan_musim.to_csv(tab_dir / "ringkasan_audit_musim.csv")

    # ==========================================
    # 3. TRANSFORMASI KOORDINAT & 3 GAMBAR AWAL (LANGKAH 2)
    # ==========================================
    coord = SkyCoord(ra=ev["ra"].values * u.deg, dec=ev["dec"].values * u.deg, frame="icrs")
    ev["gal_l"] = coord.galactic.l.deg
    ev["gal_b"] = coord.galactic.b.deg

    # Gambar 1: Histogram Energi (log10 E)
    plt.figure(figsize=(7, 4.5))
    plt.hist(ev["log10E"], bins=50, color="#1e3a8a", edgecolor="white", alpha=0.9)
    plt.xlabel(r"Energi Terekonstruksi $\log_{10}(E_{\mathrm{reco}} / \mathrm{GeV})$")
    plt.ylabel("Jumlah Event")
    plt.title("Distribusi Energi Event Kandidat Neutrino IceCube (2008–2018)")
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(fig_dir / "01_histogram_energi.png", dpi=200)
    plt.close()

    # Gambar 2: Distribusi sin(deklinasi)
    plt.figure(figsize=(7, 4.5))
    plt.hist(np.sin(np.radians(ev["dec"])), bins=40, range=(-1, 1), color="#0f766e", edgecolor="white", alpha=0.9)
    plt.xlabel(r"$\sin(\delta)$")
    plt.ylabel("Jumlah Event")
    plt.title(r"Distribusi $\sin(\delta)$ Event IceCube (Belahan Selatan vs Utara)")
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(fig_dir / "02_distribusi_sindec.png", dpi=200)
    plt.close()

    # Gambar 3: Peta Langit Event (Proyeksi Mollweide Ekuatorial)
    fig = plt.figure(figsize=(9, 4.8))
    ax = fig.add_subplot(111, projection="mollweide")
    # Konvensi: RA digeser ke [-180, 180] derajat lalu dikonversi ke radian
    x_rad = np.radians(((ev["ra"].values + 180.0) % 360.0) - 180.0)
    y_rad = np.radians(ev["dec"].values)
    ax.scatter(x_rad, y_rad, s=0.15, color="#334155", alpha=0.4, label="Event Neutrino")

    # Tandai posisi pra-registrasi TXS 0506+056 (RA ~ 77.36 deg, Dec ~ +5.69 deg)
    txs_x = np.radians(((77.36 + 180.0) % 360.0) - 180.0)
    txs_y = np.radians(5.69)
    ax.scatter([txs_x], [txs_y], s=60, color="#dc2626", marker="*", label="TXS 0506+056")

    ax.set_xlabel("Right Ascension (RA)")
    ax.set_ylabel("Deklinasi (deg)")
    ax.set_title("Peta Langit Event IceCube 10 Tahun (Proyeksi Mollweide, J2000)")
    ax.grid(True, linestyle=":", alpha=0.7)
    ax.legend(loc="upper right", markerscale=1.5)
    plt.tight_layout()
    plt.savefig(fig_dir / "03_peta_langit_mollweide.png", dpi=200)
    plt.close()

    print("\nTiga gambar awal berhasil disimpan ke results/figures/!")

Python version: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
numpy: 2.2.5
scipy: 1.15.3
pandas: 2.3.0
astropy: 8.0.3
matplotlib: 3.10.1
Checksum 10 berkas berhasil disimpan ke Logs/checksum_data_mentah.txt

--- RINGKASAN AUDIT DATA ---
Total baris & kolom: (1134450, 8)

Statistik deskriptif:
                 mjd        log10E       ang_err            ra           dec  \
count  1.134450e+06  1.134450e+06  1.134450e+06  1.134450e+06  1.134450e+06   
mean   5.663260e+04  3.726258e+00  8.507592e-01  1.798999e+02  3.395560e+00   
std    9.943157e+02  9.846252e-01  9.372278e-01  1.039954e+02  3.521732e+01   
min    5.456238e+04  1.050000e+00  2.000000e-01  0.000000e+00 -8.995700e+01   
25%    5.581732e+04  2.960000e+00  3.100000e-01  8.970925e+01 -2.108000e+01   
50%    5.665781e+04  3.250000e+00  5.800000e-01  1.797990e+02  5.666000e+00   
75%    5.748708e+04  4.660000e+00  1.030000e+00  2.699600e+02  2.719100e+01   
max    5.830797e+04  7.530000e+00  1.5